# Project – 2: Lanes Detection

Computer Vision 2025-26

Topic: Detect road lanes and check in which lane the car is riding.

Goal: Find the street lanes, detect in which lane the car is running and set an alert if a line gets crossed.

---

In [19]:
# Import the necessary packages

In [1]:
import typing as tp
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt
from ultralytics import YOLO

## Some auxiliary functions

Here some helper functions are provided.
More precisely, the following functions:
- `show_img` -> given in input and (optionally) a title it will plot the image as a matplotlib plot performing also the necessary conversion from BGR to RGB.
- `polar2cartesian` -> given an array of angles and one of radii it will return an array containing the corresponding cartesian coordinates (x,y), and an array containing the line direction (i.e., $\left[sin(\theta), -cos(\theta)\right]$).
- `draw_lines` -> given in input an image and a set of lines it will draw the lines on the image and return the resulting image.
- `fill_lines` -> given in input an image and the mask of the lines it will fill the lines with a given color and return the resulting image.
- `check_param` -> given a list of values in input check that none of them are less or equal to 0, returns a list containing the same values a part from the negative one that will be set to 1.
- `draw_dashed_line` -> given an image and two points ($pt1$ and $pt2$), it draws a dashed line between them. It calculates the total distance and does this by drawing short lines with small gaps in between.

Some of the functions were provided during the Lab2 session

In [2]:
import typing as tp
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt
from ultralytics import YOLO

def show_img(img, title=None):
    if len(img.shape) == 3 and img.shape[2] == 3:
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    plt.figure(figsize=(10,6))
    plt.imshow(img)
    if title: plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    plt.close()

def polar2cartesian(radius: np.ndarray, angle: np.ndarray) -> tp.Tuple[np.ndarray, np.ndarray]:
    return radius * np.array([np.cos(angle), np.sin(angle)]), np.array([np.sin(angle), -np.cos(angle)])

def draw_lines(img: np.ndarray, lines: np.ndarray, color: tp.List[int] = [0, 0, 255], thickness: int = 1):
    new_image = np.copy(img)
    empty_image = np.zeros(img.shape[:2], dtype=np.uint8)
    if len(lines.shape) == 1: lines = lines[None, ...]
    for rho, theta in lines:
        cx, direction = polar2cartesian(rho, theta)
        pt1 = np.round(cx + 10000*direction).astype(int)
        pt2 = np.round(cx - 10000*direction).astype(int)
        cv.line(empty_image, tuple(pt1), tuple(pt2), 255, thickness)
    mask_lines = empty_image != 0
    max_line = 0
    valid = flag = False
    for i, line in enumerate(mask_lines):
        indices = np.argwhere(line)
        if len(indices) > 1:
            flag = True
            for ii in range(len(indices)-1):
                flag = flag and indices[ii+1] == indices[ii] + 1
                if not flag:
                    valid = True
                    break
            if valid: max_line = i
        elif len(indices) == 1 and valid: max_line = i
        if flag and valid: break
    mask_boundaries = np.zeros_like(empty_image)
    mask_boundaries[max_line:] = 1
    mask = (mask_lines * mask_boundaries).astype(bool)
    new_image[mask_lines] = np.array(color)
    return new_image, mask

def fill_lines(img: np.ndarray, mask: np.ndarray, color: tp.List[int] = [0, 0, 255]):
    border = np.where(mask)
    if len(border[0]) == 0: return img, None
    possible_vertex = np.where(border[0] == np.min(border[0]))
    vertex = np.array([border[0][int(len(possible_vertex[0]) / 2)], border[1][int(len(possible_vertex[0]) / 2)]])[::-1]
    bottom_pos = [np.min(np.where(border[1] == np.min(border[1]))), np.max(np.where(border[1] == np.max(border[1])))]
    bottom_left = np.array([border[0][bottom_pos[0]], border[1][bottom_pos[0]]])[::-1]
    bottom_right = np.array([border[0][bottom_pos[1]], border[1][bottom_pos[1]]])[::-1]
    points = np.array([vertex, bottom_left, bottom_right])
    filled_img = cv.fillConvexPoly(np.copy(img), points=points, color=color)
    road_region = np.zeros(img.shape[:2], dtype=np.uint8)
    cv.fillConvexPoly(road_region, points=points, color=255)
    return filled_img, road_region

def draw_dashed_line(img, pt1, pt2, color, thickness, dash_length=20, gap_length=15):
    dist = np.sqrt((pt1[0]-pt2[0])**2 + (pt1[1]-pt2[1])**2)
    if dist == 0: return
    dashes = int(dist / (dash_length + gap_length))
    for i in range(dashes):
        start_p = (int(pt1[0] + (pt2[0]-pt1[0]) * i / dashes), int(pt1[1] + (pt2[1]-pt1[1]) * i / dashes))
        end_p = (int(pt1[0] + (pt2[0]-pt1[0]) * (i + 0.5) / dashes), int(pt1[1] + (pt2[1]-pt1[1]) * (i + 0.5) / dashes))
        cv.line(img, start_p, end_p, color, thickness)

- `classify_lines_and_regions` -> Identifies driving lanes from detected line segments, classifies them as solid or dashed( also colors them and fills the area between the lanes)

In [3]:
def classify_lines_and_regions(img: np.ndarray, lines: np.ndarray, length_threshold: int) -> np.ndarray:

    if lines is None:
        return img.copy()

    h, w = img.shape[:2]
    y_bottom = h
    y_top = int(h * 0.6)

    line_img = img.copy()
    overlay = img.copy()
    groups = []

    #Group similar lines
    for line in lines:
        x1, y1, x2, y2 = line[0]

        if y2 == y1:
            continue

        length = np.hypot(x2 - x1, y2 - y1)
        slope = (x2 - x1) / (y2 - y1)
        intercept = x1 - slope * y1

        for g in groups:
            if abs(g["slope"] - slope) < 0.15 and abs(g["intercept"] - intercept) < 60:
                g["slope"] = (g["slope"] + slope) / 2
                g["intercept"] = (g["intercept"] + intercept) / 2
                g["max_len"] = max(g["max_len"], length)
                break
        else:
            groups.append({
                "slope": slope,
                "intercept": intercept,
                "max_len": length
            })

    #Sort lines from left to right
    groups.sort(key=lambda g: g["slope"] * y_bottom + g["intercept"])

    extrapolated_lines = []

    #Draw lane lines
    for g in groups:
        x_bottom = int(g["slope"] * y_bottom + g["intercept"])
        x_top = int(g["slope"] * y_top + g["intercept"])

        pt_bottom = (x_bottom, y_bottom)
        pt_top = (x_top, y_top)
        extrapolated_lines.append([pt_bottom, pt_top])
        #Task 2.2. Detect if the line is solid or dashed
        is_solid = g["max_len"] > length_threshold
        color = (0, 255, 0) if is_solid else (0, 0, 255)

        if is_solid:
            cv.line(line_img, pt_bottom, pt_top, color, 4)
        else:
            draw_dashed_line(line_img, pt_bottom, pt_top, color, 4)

    #Fill lane regions 
    lane_colors = [
        (255, 0, 0),
        (0, 255, 255),
        (255, 0, 255),
        (0, 165, 255)
    ]
    #Task 2.3. From the street lines detect the road lanes as the regions inside each couple of lines
    for i in range(len(extrapolated_lines) - 1):
        left = extrapolated_lines[i]
        right = extrapolated_lines[i + 1]

        polygon = np.array(
            [left[0], right[0], right[1], left[1]],
            dtype=np.int32
        )

        cv.fillPoly(overlay, [polygon], lane_colors[i % len(lane_colors)])

    return cv.addWeighted(overlay, 0.3, line_img, 0.7, 0)

- `extract_road_region_kmeans` -> it uses K-means to group colors, finds the road area, and cleans the mask to get the ROI. Explanation is added in the code

In [4]:
def extract_road_region_kmeans(image: np.ndarray, k: int = 5):
    h, w = image.shape[:2]

    #Converts image to LAB and reshape for k means
    lab = cv.cvtColor(image, cv.COLOR_BGR2LAB)
    pixels = lab.reshape(-1, 3).astype(np.float32)

    #Run k-means clustering
    criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    _, labels, _ = cv.kmeans(pixels, k, None, criteria, 10, cv.KMEANS_RANDOM_CENTERS)
    labels_2d = labels.reshape(h, w)

    #Find the cluster that most likely represents the road
    best_score = -1
    road_cluster = None
    for i in range(k):
        mask = (labels_2d == i)
        size = mask.sum()
        if size < 0.05 * h * w:
            continue
        if mask[:int(0.15 * h), :].sum() > 0.05 * size:
            continue
        avg_y = np.mean(np.where(mask)[0]) / h
        if avg_y > best_score:
            best_score = avg_y
            road_cluster = i

    #If no cluster passed the filters, select the one with the lowest average y
    if road_cluster is None:
        road_cluster = max(range(k), key=lambda i: np.mean(np.where(labels_2d == i)[0]) if np.any(labels_2d == i) else 0)

    #Create a mask for the road cluster
    road_mask = (labels_2d == road_cluster).astype(np.uint8) * 255

    #Clean the mask 
    kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (11, 11))
    road_mask = cv.morphologyEx(road_mask, cv.MORPH_CLOSE, kernel)
    road_mask = cv.morphologyEx(road_mask, cv.MORPH_OPEN, kernel)

    #To keep only the largest connected component
    num_labels, cc = cv.connectedComponents(road_mask)
    if num_labels > 1:
        sizes = [np.sum(cc == i) for i in range(1, num_labels)]
        largest = np.argmax(sizes) + 1
        road_mask = (cc == largest).astype(np.uint8) * 255

    #To close gaps
    kernel_big = cv.getStructuringElement(cv.MORPH_ELLIPSE, (25, 25))
    road_mask = cv.morphologyEx(road_mask, cv.MORPH_CLOSE, kernel_big)

    #Fill holes using flood fill
    floodfill = road_mask.copy()
    f_mask = np.zeros((h + 2, w + 2), np.uint8)
    cv.floodFill(floodfill, f_mask, (0, 0), 255)
    road_mask |= cv.bitwise_not(floodfill)

    #Crop to road region
    ys, xs = np.where(road_mask > 0)
    if len(xs) == 0:
        return image, np.ones_like(road_mask) * 255, (0, 0, w, h)

    pad = 10
    x1, x2 = max(0, xs.min() - pad), min(w - 1, xs.max() + pad)
    y1, y2 = max(0, ys.min() - pad), min(h - 1, ys.max() + pad)

    road_roi = image[y1:y2, x1:x2]
    road_mask_roi = road_mask[y1:y2, x1:x2]

    final_road = np.zeros_like(road_roi)
    final_road[road_mask_roi == 255] = road_roi[road_mask_roi == 255]

    return final_road, road_mask_roi, (x1, y1, x2, y2)

- `interactive_canny` -> Helper function to adjust Canny edge detection thresholds
- `interactive_hough_road` -> Helper function to use the hough transform to detect road lines with the thresholds
- `draw_yolo_boxes` -> Draws bounding boxes around detected cars and shows a warning if a car is too close

In [5]:
def interactive_canny(img, window_name="Canny"):
    cv.namedWindow(window_name)
    cv.createTrackbar('Low thresh.', window_name, 420, 1000, lambda x: None)
    cv.createTrackbar('High thresh.', window_name, 890, 1000, lambda x: None)
    
    edges = None
    while True:
        l_t = cv.getTrackbarPos('Low thresh.', window_name)
        h_t = cv.getTrackbarPos('High thresh.', window_name)
        edges = cv.Canny(img, l_t, h_t)
        cv.imshow(window_name, edges)
        if cv.waitKey(1) & 0xFF == ord('q'): break
    cv.destroyWindow(window_name)
    return edges

#Task 2.1. Find the street lines using the Hough Transform 
def interactive_hough_road(image, edges):
    window_name = 'Hough Transform - Road Boundaries'
    cv.namedWindow(window_name)
    cv.createTrackbar('Rho', window_name, 10, 10, lambda x: None)
    cv.createTrackbar('Theta', window_name, 30, 180, lambda x: None)
    cv.createTrackbar('Threshold', window_name, 40, 200, lambda x: None)
    cv.createTrackbar('Min theta', window_name, 50, 180, lambda x: None)
    cv.createTrackbar('Max theta', window_name, 150, 180, lambda x: None)
    
    road_region_mask = None
    while True:
        rho = max(1, cv.getTrackbarPos('Rho', window_name))
        theta_val = cv.getTrackbarPos('Theta', window_name)
        theta = np.pi / theta_val if theta_val != 0 else np.pi
        h_thresh = cv.getTrackbarPos('Threshold', window_name) * 10
        m_theta = cv.getTrackbarPos('Min theta', window_name) * np.pi / 180
        mx_theta = cv.getTrackbarPos('Max theta', window_name) * np.pi / 180
        
        lines = cv.HoughLines(edges, rho, theta, h_thresh, min_theta=m_theta, max_theta=mx_theta)
        
        if lines is not None:
            lines = lines.squeeze(1)
            line_vis, mask = draw_lines(image, lines, thickness=2)
            road_vis, road_region_mask = fill_lines(image, mask)
            cv.imshow(window_name, np.hstack([line_vis, road_vis]))
        else:
            cv.imshow(window_name, image)
            
        if cv.waitKey(1) & 0xFF == ord('q'): break
    cv.destroyWindow(window_name)
    return road_region_mask

def draw_yolo_boxes(image, results, res_plot, warn_thresh=150):
    for result in results:
        for box in result.boxes:
            if box.conf[0] > 0.4:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                dist_to_ego = image.shape[0] - y2
                #Task 3. Get in which lane each car is running by matching with the previous detection
                #Get how far is the car in the other lane and set an alarm if it is close

                box_color = (0, 255, 0)
                status = "Tracking"
                
                if dist_to_ego < warn_thresh:
                    box_color = (0, 0, 255)
                    status = "Warning"
                
                cv.rectangle(res_plot, (x1, y1), (x2, y2), box_color, 2)
                label = f"{result.names[int(box.cls[0])]} | {dist_to_ego}px"
                cv.putText(res_plot, label, (x1, y1 - 25), cv.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)
                cv.putText(res_plot, status, (x1, y1 - 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)
    return res_plot

The main logic of the program:
1. Road Extraction
2. To mark the lane
3. HoughLinesP + YOLO Monitoring

In [6]:
def main(img_path: str, preprocess: bool = True, method: str = 'kmeans', k_clusters: int = 5, p_thresh: int = 40, s_d_t: int = 80):
    #Task 3.1. To use object detectors
    yolo_model = YOLO("yolov8s.pt")
    image = cv.imread(img_path)
    if image is None: return print(f"Image not found at {img_path}")

    road_region_mask = None
    if preprocess:
        if method == 'kmeans':
            road_area, road_region_mask, _ = extract_road_region_kmeans(image, k=k_clusters)
        elif method == 'hough':
            edges = interactive_canny(image, "Canny - Road Detection")
            road_region_mask = interactive_hough_road(image, edges)
            road_area = cv.bitwise_and(image, image, mask=road_region_mask) if road_region_mask is not None else image
    else:
        road_area = image.copy()

    gray_road = cv.cvtColor(road_area, cv.COLOR_BGR2GRAY)
    blur_road = cv.GaussianBlur(gray_road, (5, 5), 0)
    edges_road = interactive_canny(blur_road, "Canny - Lane Markings")
    
    if road_region_mask is not None:
        edges_road = cv.bitwise_and(edges_road, edges_road, mask=road_region_mask)

    cv.namedWindow('Final Monitoring')
    while True:
        lines_p = cv.HoughLinesP(edges_road, 1, np.pi/180, p_thresh, minLineLength=20, maxLineGap=10)
        res = classify_lines_and_regions(road_area.copy(), lines_p, s_d_t)        
        yolo_results = yolo_model(image, verbose=False)
        res = draw_yolo_boxes(image, yolo_results, res)       
        cv.imshow('Final Monitoring', res)
        if cv.waitKey(1) & 0xFF == ord('q'): break
    
    cv.destroyAllWindows()

Run with different parameters(img_path, preprocess, method, k_clusters, p_thresh, s_d_t)

The full image 1 with pre-processing with K-Means Algoritgm or using Hough Transform: Task 1.2(in this case use some pre-processing to remove the parts not linked to the road)

In [10]:
if __name__ == "__main__":
    main('road1.png', preprocess=True, method = 'kmeans', k_clusters = 10, p_thresh=100, s_d_t=200)

In [9]:
if __name__ == "__main__":
    main('road1.png', preprocess=True, method = 'hough', p_thresh=100, s_d_t=200)

Image 1 without background: Task 1.1. Start from the easier images where the background has been removed

In [7]:
if __name__ == "__main__":
    main('road1b.png', preprocess=False, p_thresh=100, s_d_t=200)

The full image 2 with pre-processing with K-Means Algoritgm or using Hough Transform: Task 1.2(in this case use some pre-processing to remove the parts not linked to the road)

In [11]:
if __name__ == "__main__":
    main('road2.jpg', preprocess=True, method = 'kmeans', k_clusters = 2, p_thresh=60, s_d_t=80)

In [12]:
if __name__ == "__main__":
    main('road2.jpg', preprocess=True, method = 'hough', p_thresh=60, s_d_t=80)

Image 2 without background: Task 1.1. Start from the easier images where the background has been removed

In [13]:
if __name__ == "__main__":
    main('road2b.jpg', preprocess=False, p_thresh=60, s_d_t=80)

The full image 3 with pre-processing with K-Means Algoritgm or using Hough Transform: Task 1.2(in this case use some pre-processing to remove the parts not linked to the road)

In [14]:
if __name__ == "__main__":
    main('road3.jpg', preprocess=True, method = 'kmeans', k_clusters = 4, p_thresh=40, s_d_t=80)

In [15]:
if __name__ == "__main__":
    main('road3.jpg', preprocess=True, method = 'hough', p_thresh=40, s_d_t=80)

Image 3 without background: Task 1.1. Start from the easier images where the background has been removed

In [16]:
if __name__ == "__main__":
    main('road3b.jpg', preprocess=False, p_thresh=40, s_d_t=80)

The full image 4 with pre-processing with K-Means Algoritgm or using Hough Transform: Task 1.2(in this case use some pre-processing to remove the parts not linked to the road)

In [17]:
if __name__ == "__main__":
    main('road4.png', preprocess=True, method = 'kmeans', k_clusters = 2, p_thresh=45, s_d_t=140)

In [18]:
if __name__ == "__main__":
    main('road4.png', preprocess=True, method = 'hough', p_thresh=45, s_d_t=140)

Image 4 without background: Task 1.1. Start from the easier images where the background has been removed

In [19]:
if __name__ == "__main__":
    main('road4b.png', preprocess=False, p_thresh=45, s_d_t=140)